In [1]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from _collections_abc import Sequence
from operator import add

In [2]:
class OverAllState(TypedDict):
    inputs: list[str]
    entries: Annotated[list[tuple[str, int]], add]
    word_counts: dict[str, int]

class MapInputState(TypedDict):
    sentence: str

def map_router(state: OverAllState) -> Sequence[Send]:
    inputs = state["inputs"]

    return [
        Send(
            node="mapper_node",
            arg={
                "sentence": sentence
            }
        ) for sentence in inputs
    ]

def mapper_node(state: MapInputState) -> OverAllState:
    sentence = state["sentence"]

    words = sentence.split(" ")

    entries = [(word, 1) for word in words]

    return {
        "entries": entries
    }

def reducer_node(state: OverAllState) -> OverAllState:
    entries = state["entries"]

    word_counts = {}
    for k, v in entries:
        word_counts[k] = word_counts.get(k, 0) + v

    return {
        "word_counts": word_counts
    }

builder = StateGraph(state_schema=OverAllState)

builder.add_node(mapper_node)
builder.add_node(reducer_node)

builder.add_conditional_edges(
    START,
    map_router,
    ["mapper_node"]
)

builder.add_edge("mapper_node", "reducer_node")
builder.add_edge("reducer_node", END)

graph = builder.compile()

res = graph.invoke({
    "inputs": ["hello world", "hello atguigu", "hello LLM"]
})

print(res)

{'inputs': ['hello world', 'hello atguigu', 'hello LLM'], 'entries': [('hello', 1), ('world', 1), ('hello', 1), ('atguigu', 1), ('hello', 1), ('LLM', 1)], 'word_counts': {'hello': 3, 'world': 1, 'atguigu': 1, 'LLM': 1}}
